## This Jupyter notebook is used for analysis sentimental classficiation results of 3 different LLMs under Few-Shot-Learning (FSL) and Semi-Surpervised Learning (SSL|in record log folder will be marked as SSP) task. To analysis, understand, assume, and propose solution.

In [1]:
import torch
from torch import nn, Tensor, functional
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('../')

In [2]:
from predict_label_output import find_experiment_dirs, EMOTION_LIST
import re, os, json
from typing import Dict, List, Any

/home/new/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [25]:
qualifed_dirs = find_experiment_dirs(outputs_root='../outputs')
qualifed_dirs = ['../outputs/Qwen3_02-11/method_FSL_bs_6_inputdata_ws_prompt_Qwen4b_FSL32', '../outputs/Llama_02-11/method_FSL_bs_4_inputdata_ws_prompt_Llama3.3_FSL24', '../outputs/Qwen3_02-11/method_SSP_bs_16_inputdata_ws_prompt_Qwen1d7_SSP_03', '../outputs/Qwen3_02-11/method_SSP_bs_6_inputdata_ws_prompt_Qwen1d7_SSP_01']
regex_code = r"predictions?_step\d+_\w+\.json"

def has_matching_file(dir, pattern=regex_code):
    regex = re.compile(pattern)
    for filename in os.listdir(dir):
        if regex.match(filename):
            return True, filename
    return False, None

Found 11 completed experiments


In [26]:
# Store pairs of (final_results.json, predictions_file.json)
selected_dirs = list()
for dirs in qualifed_dirs:
    matched, fname = has_matching_file(dirs)
    if matched:
        pred_file = os.path.join(dirs, fname)
        final_results_file = os.path.join(dirs, 'final_results.json')
        
        # Only add if final_results.json exists
        if os.path.exists(final_results_file):
            selected_dirs.append((final_results_file, pred_file))
        else:
            print(f"Warning: final_results.json not found in {dirs}")

In [5]:
def json_read(file_path):
    """Read JSON file and return parsed content"""
    with open(file_path, 'r', encoding='utf-8') as file:
        jsonf = json.load(file)
    return jsonf

In [6]:
def conclude_detail(texts: Dict, emo_dict: Dict[str, Dict]):
    for idx, text in enumerate(texts):
        is_correct = text['is_correct']
        target, pred = text['target_emotion'], text['pred_emotion']
        if is_correct:
            emo_dict[target][1]+=1
            continue
        emo_dict[target][0][pred]+=1
    return emo_dict

In [7]:
def str2bool(givenw: str):
    if re.match(r'Trues?', givenw, re.IGNORECASE):
        return True
    elif re.match(r'Falses?', givenw, re.IGNORECASE):
        return False
    return None

In [8]:
def extract_intel_from_final_results(final_results_path: str) -> Dict:
    """
    Extract model information and training strategy from final_results.json
    
    Returns:
        Dict with keys: model_name, learning_strategy, learning_ratio
    """
    final_results = json_read(final_results_path)
    config = final_results.get('config', {})
    
    # Extract model name (simplified)
    model_name = config.get('model_name', 'Unknown')
    if 'Llama' in model_name:
        model_name = 'Llama3'
    elif 'Qwen' in model_name:
        if '1.7' in model_name or '1d7' in model_name:
            model_name = 'Qwen1.7B'
        elif '4' in model_name:
            model_name = 'Qwen4B'
        else:
            model_name = 'Qwen'
    
    # Determine learning strategy
    few_shot = config.get('few_shot', False)
    semi_supervised = config.get('semi_supervised', False)
    
    if few_shot:
        learning_strategy = 'FSL'
        learning_ratio = config.get('shots_per_class', 'N/A')
        learning_info = f"FSL-{learning_ratio}"
    elif semi_supervised:
        learning_strategy = 'SSL'
        learning_ratio = config.get('semi_ratio', 'N/A')
        learning_info = f"SSL-{learning_ratio}"
    else:
        learning_strategy = 'Full'
        learning_ratio = 'N/A'
        learning_info = 'Full Training'
    
    return {
        'model_name': model_name,
        'learning_strategy': learning_strategy,
        'learning_ratio': learning_ratio,
        'learning_info': learning_info,
        'config': config
    }


def draw_the_heatmap(emo_dict: Dict[str, Dict], intel: Dict):
    """
    Draw a 32x32 confusion matrix heatmap showing misclassification patterns
    
    Args:
        emo_dict: Dictionary with structure {emotion: {1: correct_count, 0: {other_emotion: wrong_count}}}
        intel: Dictionary with model info (model_name, learning_strategy, learning_ratio)
    """
    # Create 32x32 confusion matrix
    confusion_matrix = np.zeros((32, 32), dtype=int)
    
    for target_idx, target_emo in enumerate(EMOTION_LIST):
        if target_emo not in emo_dict:
            continue
        
        emo_stats = emo_dict[target_emo]
        
        # Correct predictions go on diagonal
        correct_count = emo_stats.get(1, 0)
        confusion_matrix[target_idx, target_idx] = correct_count
        
        # Misclassifications
        misclass_dict = emo_stats.get(0, {})
        for pred_emo, count in misclass_dict.items():
            pred_idx = EMOTION_LIST.index(pred_emo)
            confusion_matrix[target_idx, pred_idx] = count
    
    # Create figure
    plt.figure(figsize=(20, 18))
    
    # Draw heatmap
    sns.heatmap(
        confusion_matrix, 
        annot=False,  # Don't show numbers (too crowded for 32x32)
        fmt='d',
        cmap='YlOrRd',  # Yellow to Orange to Red - higher = more misclassification
        xticklabels=EMOTION_LIST,
        yticklabels=EMOTION_LIST,
        cbar_kws={'label': 'Count'},
        square=True,
        linewidths=0.5,
        linecolor='gray'
    )
    
    # Labels and title
    plt.xlabel('Predicted Emotion', fontsize=14, fontweight='bold')
    plt.ylabel('Target Emotion', fontsize=14, fontweight='bold')
    
    title = f"Emotion Misclassification Heatmap\n"
    title += f"Model: {intel['model_name']} | Strategy: {intel['learning_info']}"
    plt.title(title, fontsize=16, fontweight='bold', pad=20)
    
    plt.xticks(rotation=45, ha='right', fontsize=10)
    plt.yticks(rotation=0, fontsize=10)
    plt.tight_layout()
    
    # Show
    plt.show()
    
    # Calculate and display statistics
    total_samples = confusion_matrix.sum()
    correct_predictions = np.diag(confusion_matrix).sum()
    accuracy = correct_predictions / total_samples if total_samples > 0 else 0
    
    print(f"\n{'='*70}")
    print(f"Model: {intel['model_name']}")
    print(f"Learning Strategy: {intel['learning_info']}")
    print(f"Total Samples: {total_samples}")
    print(f"Correct Predictions: {correct_predictions}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"{'='*70}\n")


def collect_intel(final_results_path: str, predictions_path: str):
    """
    Main function to collect intel and draw heatmap
    
    Args:
        final_results_path: Path to final_results.json
        predictions_path: Path to predictions_stepXXXX_val.json
    """
    # Load intel from final_results.json
    intel = extract_intel_from_final_results(final_results_path)
    
    # Load predictions
    pred_data = json_read(predictions_path)
    
    # Initialize emo_dict
    emo_dict = dict()
    for emo in EMOTION_LIST:
        inner_dict = {side_emo: 0 for side_emo in EMOTION_LIST if side_emo != emo}
        stastic_dict = {1: 0, 0: inner_dict}
        emo_dict[emo] = stastic_dict
    
    # Fill emo_dict with prediction results
    emo_dict = conclude_detail(pred_data['text_output'], emo_dict)
    
    # Draw the heatmap
    draw_the_heatmap(emo_dict, intel)

## Top-k selection drawing

Data pre-colleciton part, from emo_dict

In [27]:
def collect_all_misclassifications(selected_dirs: List[tuple]) -> Dict:
    """
    遍历所有实验，按照模型和训练策略分组收集误分类数据
    
    Args:
        selected_dirs: List of (final_results_path, predictions_path) tuples
        
    Returns:
        grouped_data: {
            'Llama3': {
                'FSL': {(target, pred): count, ...},
                'SSL': {(target, pred): count, ...}
            },
            'Qwen4B': {...},
            'Qwen1.7B': {...}
        }
    """
    # 初始化数据结构
    model_names = ['Llama3', 'Qwen4B', 'Qwen1.7B']
    # strategies = ['FSL', 'SSL']
    
    grouped_data = {}
    for model in model_names:
        grouped_data[model] = {}
        # for strategy in strategies:
        #     grouped_data[model][strategy] = {}
    
    # 遍历所有实验
    print(f"开始收集 {len(selected_dirs)} 个实验的误分类数据...")
    
    for final_results_path, predictions_path in selected_dirs:
        try:
            # 提取实验信息
            intel = extract_intel_from_final_results(final_results_path)
            model = intel['model_name']
            # strategy = intel['learning_strategy']
            strategy = intel['learning_info']
            if strategy == 'Full Training': continue
            grouped_data[model][strategy] = dict()
            grouped_data[model]['filepath'] = predictions_path
            
            # 跳过不在预期范围内的模型或策略
            if model not in model_names:
                print(f"⚠️  跳过: {model} - {strategy} (不在目标范围)")
                continue
            
            # 加载预测数据并构建emo_dict
            pred_data = json_read(predictions_path)
            
            # 初始化emo_dict
            emo_dict = {}
            for emo in EMOTION_LIST:
                inner_dict = {side_emo: 0 for side_emo in EMOTION_LIST if side_emo != emo}
                emo_dict[emo] = {1: 0, 0: inner_dict}
            
            # 填充emo_dict
            emo_dict = conclude_detail(pred_data['text_output'], emo_dict)
            
            # 收集所有误分类对并累加到grouped_data
            for target_emo in EMOTION_LIST:
                misclass_dict = emo_dict[target_emo][0]
                for pred_emo, count in misclass_dict.items():
                    if count > 0:  # 只记录有误分类的
                        key = (target_emo, pred_emo)
                        grouped_data[model][strategy][key] = \
                            grouped_data[model][strategy].get(key, 0) + count
            
            print(f"✓ {model}-{strategy}: 已处理")
            
        except Exception as e:
            print(f"✗ 处理失败 {predictions_path}: {e}")
            continue
    
    # 打印统计信息
    print(f"\n{'='*70}")
    print("数据收集完成! 统计信息:")
    
    return grouped_data


def select_topk_misclassifications(grouped_data: Dict, k: int = 10) -> Dict:
    """
    从分组数据中选取每个模型-策略组合的top-k误分类
    
    Args:
        grouped_data: collect_all_misclassifications()的输出
        k: 选取前k个误分类 (默认10)
        
    Returns:
        topk_data: {
            'Llama3': {
                'FSL': [(target, pred, count), ...],  # 最多k条 + 1条Others
                'SSL': [(target, pred, count), ...]
            },
            ...
        }
    """
    topk_data = {}
    
    for model in grouped_data:
        topk_data[model] = {}
        
        for strategy in grouped_data[model]:
            if isinstance(grouped_data[model][strategy], str): continue
            # 将字典转换为列表: [(target, pred, count), ...]
            misclass_list = [
                (target, pred, count)
                for (target, pred), count in grouped_data[model][strategy].items()
            ]
            
            # 按count降序排序
            misclass_list.sort(key=lambda x: x[2], reverse=True)
            
            # 选取top-k
            top_k_items = misclass_list[:k]
            
            # 计算剩余误分类的统计数据
            remaining_items = misclass_list[k:k+100]
            if len(remaining_items) > 0:
                # 计算剩余项的平均值
                total_remaining = sum(item[2] for item in remaining_items)
                avg_remaining = total_remaining / len(remaining_items)
                
                # 添加"Others"条目
                # target: 'Others', pred: 'Avg (N=XX)'
                others_entry = ('Others', f'Avg (Rest={len(remaining_items)})', int(avg_remaining))
                top_k_items.append(others_entry)
                
            topk_data[model][strategy] = top_k_items
    
    # 打印top-k结果概览
    print(f"\n{'='*70}")
    print(f"Top-{k} 误分类选取完成 (包含Others统计)!")
    print(f"{'='*70}")
    for model in topk_data:
        for strategy in topk_data[model]:
            topk_list = topk_data[model][strategy]
            print(f"\n{model} - {strategy}:")
            if len(topk_list) > 0:
                # 检查最后一个是否是Others
                has_others = topk_list[-1][0] == 'Others'
                real_count = len(topk_list) - 1 if has_others else len(topk_list)
                
                print(f"  共 {len(topk_list)} 条显示记录")
                print(f"  最高误分类: {topk_list[0][0]} → {topk_list[0][1]} ({topk_list[0][2]}次)")
                
                if has_others:
                    others = topk_list[-1]
                    print(f"  剩余类别: {others[2]:.2f} (平均次数), 包含 {others[1]}")
            else:
                print(f"  ⚠️  无数据")
    print(f"{'='*70}\n")
    
    return topk_data


def validate_topk_data(topk_data: Dict) -> bool:
    """
    验证top-k数据的完整性和正确性
    
    Args:
        topk_data: select_topk_misclassifications()的输出
        
    Returns:
        bool: 数据是否有效
    """
    print("数据验证中...")
    
    all_valid = True
    
    for model in topk_data:
        for strategy in topk_data[model]:
            
            data_list = topk_data[model][strategy]
            
            # 检查1: 允许超过10条记录 (因为添加了Others)
            if len(data_list) > 15: # 放宽限制
                print(f"✗ {model}-{strategy}: 记录过多 ({len(data_list)})")
                all_valid = False
            
            # 检查2: count值降序排列 (Top-k部分)
            # 忽略最后一个Others(如果存在)
            check_len = len(data_list)
            if check_len > 0 and data_list[-1][0] == 'Others':
                check_len -= 1
                
            for i in range(check_len - 1):
                if data_list[i][2] < data_list[i+1][2]:
                    print(f"✗ {model}-{strategy}: count值未降序排列 (Index {i})")
                    all_valid = False
                    break
            
            # 检查3: target和pred都在EMOTION_LIST中 (忽略Others)
            for target, pred, count in data_list:
                if target == 'Others': continue # 跳过Others检查
                
                if target not in EMOTION_LIST or pred not in EMOTION_LIST:
                    print(f"✗ {model}-{strategy}: 情感标签不在列表中 ({target}, {pred})")
                    all_valid = False
    
    if all_valid:
        print("✓ 数据验证通过!\n")
    else:
        print("✗ 数据验证失败!\n")
    
    return all_valid

## Usage: Collect Top-K Misclassification Data

Run this to collect and validate the top-10 misclassifications for each model-strategy combination:

In [28]:
# Step 1: Collect all misclassifications grouped by model and strategy
grouped_data = collect_all_misclassifications(selected_dirs)

# Step 2: Select top-10 for each (model, strategy) combination
topk_data = select_topk_misclassifications(grouped_data, k=10)

# Step 3: Validate the data
is_valid = validate_topk_data(topk_data)

# Show the structure
print("\n数据结构预览:")
print(f"Models: {list(topk_data.keys())}")
for model in topk_data:
    print(f"\n{model}:")
    for strategy in topk_data[model]:
        data = topk_data[model][strategy]
        print(f"  {strategy}: {len(data)} 条记录")
        if len(data) > 0:
            print(f"    示例: {data[0]}")

开始收集 4 个实验的误分类数据...
✓ Qwen4B-FSL-32: 已处理
✓ Llama3-FSL-24: 已处理
✓ Qwen1.7B-SSL-0.5: 已处理
✓ Qwen4B-SSL-0.4: 已处理

数据收集完成! 统计信息:

Top-10 误分类选取完成 (包含Others统计)!

Llama3 - FSL-24:
  共 11 条显示记录
  最高误分类: terrified → afraid (94次)
  剩余类别: 11.00 (平均次数), 包含 Avg (Rest=100)

Qwen4B - FSL-32:
  共 11 条显示记录
  最高误分类: angry → furious (70次)
  剩余类别: 16.00 (平均次数), 包含 Avg (Rest=100)

Qwen4B - SSL-0.4:
  共 11 条显示记录
  最高误分类: angry → furious (52次)
  剩余类别: 6.00 (平均次数), 包含 Avg (Rest=100)

Qwen1.7B - SSL-0.5:
  共 11 条显示记录
  最高误分类: afraid → terrified (39次)
  剩余类别: 5.00 (平均次数), 包含 Avg (Rest=100)

数据验证中...
✓ 数据验证通过!


数据结构预览:
Models: ['Llama3', 'Qwen4B', 'Qwen1.7B']

Llama3:
  FSL-24: 11 条记录
    示例: ('terrified', 'afraid', 94)

Qwen4B:
  FSL-32: 11 条记录
    示例: ('angry', 'furious', 70)
  SSL-0.4: 11 条记录
    示例: ('angry', 'furious', 52)

Qwen1.7B:
  SSL-0.5: 11 条记录
    示例: ('afraid', 'terrified', 39)


## Drawing Part: Top-K Misclassification Visualization

Visualize the top-10 misclassifications for each model and strategy:

In [29]:
def draw_topk_misclassification_comparison(topk_data: Dict, grouped_data: Dict = None):
    """
    绘制Top-K误分类对比图 (堆叠图)
    
    布局: N行2列，每个策略设置单独一个subplot
    X轴: target emotion（真实标签）
    Y轴: 误分类次数
    
    每个柱子由两部分组成:
    1. 下半部分 (实色): Target -> Pred 的误分类次数 (Forward)
    2. 上半部分 (透明/浅色): Pred -> Target 的误分类次数 (Reverse/Inverse) - 仅当 grouped_data 提供时显示
    
    Args:
        topk_data: {model: {strategy_info: [(target, pred, count), ...]}}
        grouped_data: (Optional) 原始完整数据，用于查找反向误分类数量 (Pred->Target)。
                      结构: {model: {strategy_info: {(target, pred): count}}}
    """
    # 配置颜色
    fsl_colors = ['#3498db', '#5dade2', '#85c1e9', '#aed6f1']  # 蓝色系 (Bottom)
    fsl_top_colors = ['#AED6F1', '#D6EAF8'] # 浅蓝色 (Top)
    
    ssl_colors = ['#e74c3c', '#f1948a', '#f5b7b1', '#fadbd8']  # 红色系 (Bottom)
    ssl_top_colors = ['#F5B7B1', '#FADBD8'] # 浅红色 (Top)
    
    # 扁平化数据: 创建 (model, strategy_info, data) 列表
    plot_items = []
    models = list(topk_data.keys())
    for model in models:
        for strategy_info, data in topk_data[model].items():
            plot_items.append((model, strategy_info, data))
    
    # 计算网格大小
    num_plots = len(plot_items)
    num_rows = num_plots // 2 + (1 if num_plots % 2 != 0 else 0)
    
    # 创建图表
    fig, axes = plt.subplots(num_rows, 2, figsize=(28, 7 * num_rows))
    fig.suptitle('Top-10 Misclassification Analysis (Stacked: Target->Pred + Pred->Target)', 
                 fontsize=24, fontweight='bold', y=0.998)
    
    # 确保axes是2D数组
    if num_rows == 1:
        axes = axes.reshape(1, -1)
    elif num_plots == 1:
        # 特殊情况处理
        axes = np.array([[axes]])
    
    # 绘制每个subplot
    for idx, (model, strategy_info, data) in enumerate(plot_items):
        row = idx // 2
        col = idx % 2
        ax = axes[row, col]
        
        # 获取该策略的完整数据以查找反向对
        strategy_full_data = None
        if grouped_data and model in grouped_data and strategy_info in grouped_data[model]:
             strategy_full_data = grouped_data[model][strategy_info]
        
        # 根据策略类型选择颜色
        if strategy_info.startswith('FSL'):
            color_bottom = fsl_colors[0]
            color_top = fsl_top_colors[0]
        elif strategy_info.startswith('SSL'):
            color_bottom = ssl_colors[0]
            color_top = ssl_top_colors[0]
        else:
            color_bottom = '#95a5a6'  # 灰色
            color_top = '#D7DBDD'     # 浅灰色
        
        # 绘制单个策略的数据
        _draw_single_strategy_subplot(ax, data, strategy_full_data, color_bottom, color_top, f'{model} - {strategy_info}')
    
    # 隐藏多余的subplot
    for idx in range(num_plots, num_rows * 2):
        row = idx // 2
        col = idx % 2
        axes[row, col].axis('off')
    
    plt.tight_layout()
    plt.subplots_adjust(top=0.92, hspace=0.4) # 调整顶部空间给大标题
    plt.show() # Note: In Jupyter, this shows the plot.
    
    print("\n" + "="*70)
    print("Stack Visualization Complete!")
    print("="*70)
    print("  - Bottom Bar (Dark): T -> P (Target emotion misclassified as Pred)")
    print("  - Top Bar (Light/Hatched): P -> T (Pred emotion misclassified as Target) [Reverse Confusion]")
    print(f"  - Total {num_plots} strategies visualized.")
    print("="*70 + "\n")


def _draw_single_strategy_subplot(ax, data_list, full_data, color_bottom, color_top, title):
    """
    在单个subplot中绘制一个策略的误分类 (堆叠)
    """
    if len(data_list) == 0:
        ax.text(0.5, 0.5, 'No Data', ha='center', va='center', 
               fontsize=18, color='gray')
        ax.set_title(title, fontsize=16, fontweight='bold')
        ax.axis('off')
        return
    
    # 准备数据
    targets = []
    preds = []
    counts_forward = [] # A -> B
    counts_reverse = [] # B -> A
    
    for item in data_list:
        t, p, c = item[0], item[1], item[2]
        targets.append(t)
        preds.append(p)
        counts_forward.append(c)
        
        # 查找反向混淆: Pred -> Target
        rev_count = 0
        if t != 'Others' and full_data:
            # 只有当完整数据存在且不是 'Others' 类别时才查找反向
            rev_key = (p, t) # Switch key
            rev_count = full_data.get(rev_key, 0)
        
        counts_reverse.append(rev_count)
    
    x_positions = list(range(len(data_list)))
    
    # 1. 绘制 Bottom Bars (Forward: Target -> Pred)
    bars_bottom = ax.bar(x_positions, counts_forward, color=color_bottom, 
                 width=0.7, edgecolor='black', linewidth=1.2, alpha=0.9, label='Target->Pred')
    
    # 2. 绘制 Top Bars (Reverse: Pred -> Target) - 堆叠在 Bottom 之上
    # 使用 hatch (阴影线) 来区分
    bars_top = ax.bar(x_positions, counts_reverse, bottom=counts_forward, color=color_top,
                 width=0.7, edgecolor='black', linewidth=1.2, alpha=0.7, hatch='//', label='Pred->Target')
    
    # 计算最大高度用于标签定位
    total_heights = [b + t for b, t in zip(counts_forward, counts_reverse)]
    max_height = max(total_heights) if total_heights else 1
    
    # 添加标签
    for i, (bar_b, bar_t, pred_label, c_fwd, c_rev) in enumerate(zip(bars_bottom, bars_top, preds, counts_forward, counts_reverse)):
        
        # A. 底部数值 (Forward count)
        # 如果柱子够高，显示在中间；否则不显示或显示在上方(如果top为0)
        if c_fwd > max_height * 0.1:
            ax.text(bar_b.get_x() + bar_b.get_width()/2, c_fwd/2, f'{c_fwd}', 
                   ha='center', va='center', fontsize=10, color='white', fontweight='bold')
        
        # B. 顶部数值 (Reverse count) - 仅当 > 0 时显示
        if c_rev > 0:
            # 显示位置: bottom height + half of top height
            label_pos = c_fwd + c_rev/2
            if c_rev > max_height * 0.1: # 只有足够空间才在内部显示
                ax.text(bar_t.get_x() + bar_t.get_width()/2, label_pos, f'{c_rev}', 
                       ha='center', va='center', fontsize=10, color='black', alpha=0.7)
            else:
                 # 空间不够，不显示数值，或者可以显示在旁边，这里选择简化不显示微小数值
                 pass

        # C. 顶部总数值或者Pred标签
        # 标签位置
        total_h = c_fwd + c_rev
        label_y = total_h + max_height * 0.02
        
        # 显示预测标签 (Pred Emotion)
        ax.text(bar_b.get_x() + bar_b.get_width()/2, label_y, pred_label, 
               ha='center', va='bottom', fontsize=11, fontweight='bold', 
               rotation=45, color='darkblue')

    # 设置x轴刻度和标签（target情感）
    ax.set_xticks(x_positions)
    # 对于 Others, pred 是 'Avg...', 我们只显示 Target ‘Others’
    xtick_labels = [f"{t}" for t in targets]
    ax.set_xticklabels(xtick_labels, rotation=45, ha='right', fontsize=11)
    
    # 设置标题和标签
    ax.set_title(title, fontsize=16, fontweight='bold', pad=15)
    ax.set_xlabel('Target Emotion', fontsize=12, fontweight='bold')
    if data_list[0][0] == targets[0]: # Limit y-label to left-most plots logically or just all
        ax.set_ylabel('Count (Bottom: T->P, Top: P->T)', fontsize=12, fontweight='bold')
    
    # 添加图例 (只在第一个图或者每个图都加)
    # ax.legend(loc='upper right', fontsize=8) 
    
    # 添加网格
    ax.grid(axis='y', linestyle='--', alpha=0.5, linewidth=1.0)
    ax.set_axisbelow(True)
    ax.set_ylim(bottom=0)
    
    # 自动调整Y轴上限，留出标签空间
    ax.set_ylim(top=max_height * 1.35) 

In [30]:
def print_topk_misclassifications(topk_data: Dict):
    """
    Print top-k misclassification data in a formatted text output
    
    Format:
    =====Model - Strategy=====
    target, pred, count
    emotion1, emotion2, 10
    ...
    ===============
    
    Args:
        topk_data: {model: {strategy_info: [(target, pred, count), ...]}}
    """
    print("\n" + "="*70)
    print("Top-K Misclassification Data (Formatted Output)")
    print("="*70 + "\n")
    
    for model in sorted(topk_data.keys()):
        for strategy_info in sorted(topk_data[model].keys()):
            data_list = topk_data[model][strategy_info]
            
            # Print header
            print(f"====={model} - {strategy_info}=====")
            print("target, pred, count")
            
            # Print data rows
            if len(data_list) > 0:
                for target, pred, count in data_list:
                    print(f"{target}, {pred}, {count}")
            else:
                print("(No data)")
            
            print("\n===============\n")
    
    print("="*70)
    print("End of Top-K Misclassification Report")
    print("="*70 + "\n")

## Print Top-K Data in Text Format

Print the top-k misclassifications in a formatted text output:

In [31]:
# Print the top-k data in formatted text
if 'topk_data' in locals():
    print_topk_misclassifications(topk_data)
else:
    print("⚠️  Please run the data collection cells first to generate 'topk_data'")
    print("   Execute cell 15 to collect the data before running this print function.")


Top-K Misclassification Data (Formatted Output)

=====Llama3 - FSL-24=====
target, pred, count
terrified, afraid, 94
anticipating, excited, 77
angry, furious, 74
guilty, ashamed, 55
anxious, afraid, 47
sad, devastated, 43
afraid, terrified, 41
nostalgic, sentimental, 39
proud, impressed, 38
lonely, sad, 37
Others, Avg (Rest=100), 11


=====Qwen1.7B - SSL-0.5=====
target, pred, count
afraid, terrified, 39
angry, furious, 37
nostalgic, sentimental, 29
anxious, anticipating, 23
excited, anticipating, 22
angry, annoyed, 21
furious, angry, 19
sad, devastated, 18
excited, joyful, 17
guilty, ashamed, 17
Others, Avg (Rest=100), 5


=====Qwen4B - FSL-32=====
target, pred, count
angry, furious, 70
impressed, proud, 63
angry, annoyed, 60
surprised, joyful, 58
sad, devastated, 52
apprehensive, anxious, 50
hopeful, confident, 50
terrified, afraid, 49
joyful, content, 48
prepared, confident, 46
Others, Avg (Rest=100), 16


=====Qwen4B - SSL-0.4=====
target, pred, count
angry, furious, 52
afraid, te

In [32]:
import os

# Create a copy to avoid modifying the original during iteration if needed
example_dict = grouped_data.copy()
extracted_data_storage = []

for model in example_dict:
    # 1. Safely get the filepath. It's stored directly under the model key.
    # The original loop `for k, v in grouped_data[model]` would iterate keys ('filepath', strategy_name), 
    # and `fpred` might not be set before we try to use it if 'filepath' comes later in iteration.
    fpred = example_dict[model].get('filepath')
    
    if not fpred:
        print(f"Skipping model {model}: No filepath found.")
        continue

    # 2. Iterate over strategies. We need to skip the 'filepath' key itself.
    for strategy_name, misclass_data in example_dict[model].items():
        if strategy_name == 'filepath': continue
        
        # misclass_data is a dict: {(target, pred): count, ...}
        
        # 3. Sort by fault count (highest to lowest). 
        # Original code `sorted(v.items(), key = lambda item: item[1])` sorts ascending (lowest to highest).
        # We need reverse=True for "height to lower".
        sorted_items = sorted(misclass_data.items(), key=lambda item: item[1], reverse=True)
        
        # Initialize output_dict to hold the detailed text samples for each error type
        # We use the keys from our sorted list to ensure we cover all errors
        output_dict = {error_key: [] for error_key, count in sorted_items}
        
        # Read the full predictions file to get the texts
        current_json = json_read(fpred)
        # tsamples = current_json['total_samples']
        txt_output = current_json['text_output']
        extracted_data_storage.append((output_dict, current_json, fpred))
    
        # Populate output_dict with the actual text samples
        for txt in txt_output:
            # Skip correct predictions
            # The JSON file has "is_correct": false for errors
            if txt.get('is_correct', True): continue
            
            tar = txt.get('target_emotion')
            pred = txt.get('pred_emotion')
            
            # Only store if this specific error (tar, pred) was in our grouped_data
            # (It should be, but good to be safe)
            key = (tar, pred)
            if key in output_dict:
                output_dict[key].append(txt)
                
        # 4. Write to file
        # We want to name the file based on the model and strategy
        # Original code used `k` from the loop which was `strategy_name`
        file_dir = os.path.dirname(fpred)
        file_name = f'predic_fault_sorted_{model}_{strategy_name}.txt'
        file_path = os.path.join(file_dir, file_name)
        
        print(f"Writing sorted faults for {model} - {strategy_name} to: {file_path}")
        
        with open(file_path, mode='w', encoding='utf-8') as f:
            # 5. Iterate through `sorted_items` to maintain the SORTED order
            # Original code `for k, v in output_dict.items():` would iterate in insertion order 
            # (which might be consistent but safer to use the explicitly sorted list)
            new_idx = 0
            for (target_emo, pred_emo), count in sorted_items:
                samples = output_dict.get((target_emo, pred_emo), [])
                
                f.write(f"=== Misclassification: {target_emo} -> {pred_emo} (Count: {count}) ===\n")
                
                for value in samples:
                    # 6. Safe unpacking. The JSON keys are explicit.
                    # Based on your file read: "sample_idx", "target_emotion", "pred_emotion", "is_correct", "text"
                    sample_idx = value.get('sample_idx', 'N/A')
                    text = value.get('text', '')
                    
                    f.write(f"[Sample {sample_idx}] -- [New idx {new_idx}]\n")
                    f.write(f"[target emotion: {target_emo} -- pred emotion: {pred_emo}]")
                    f.write(" ✗\n")
                    f.write(f"[Input Text]\n{text}\n")
                    f.write("-" * 70 + "\n\n")
                    new_idx+=1

Writing sorted faults for Llama3 - FSL-24 to: ../outputs/Llama_02-11/method_FSL_bs_4_inputdata_ws_prompt_Llama3.3_FSL24/predic_fault_sorted_Llama3_FSL-24.txt
Writing sorted faults for Qwen4B - FSL-32 to: ../outputs/Qwen3_02-11/method_SSP_bs_6_inputdata_ws_prompt_Qwen1d7_SSP_01/predic_fault_sorted_Qwen4B_FSL-32.txt
Writing sorted faults for Qwen4B - SSL-0.4 to: ../outputs/Qwen3_02-11/method_SSP_bs_6_inputdata_ws_prompt_Qwen1d7_SSP_01/predic_fault_sorted_Qwen4B_SSL-0.4.txt
Writing sorted faults for Qwen1.7B - SSL-0.5 to: ../outputs/Qwen3_02-11/method_SSP_bs_16_inputdata_ws_prompt_Qwen1d7_SSP_03/predic_fault_sorted_Qwen1.7B_SSL-0.5.txt


In [33]:
def extract_dialogue_blocks(output_dict: Dict, full_txt_output: List[Dict]) -> Dict:
    """
    For each mismatched sample in output_dict, find its entire dialogue block.
    A block is defined as consecutive samples in full_txt_output sharing the same target_emotion.
    
    Args:
        output_dict: Dictionary with keys (target, pred) and values [sample_dicts]
        full_txt_output: List of all samples from the original JSON file
        
    Returns:
        block_dict: Dictionary with same keys as output_dict, containing lists of blocks.
                    Each block is a list of sample dicts (consecutive turns).
    """
    block_dict = {key: [] for key in output_dict.keys()}
    
    # Pre-calculate mapping from sample_idx to list index to ensure accuracy
    # This handles cases where sample_idx is not 0-based sequential or data is filtered
    idx_map = {item.get('sample_idx'): i for i, item in enumerate(full_txt_output)}
    total_samples = len(full_txt_output)

    for error_key, mismatched_samples in output_dict.items():
        # error_key is (target_emotion, pred_emotion)
        
        for sample in mismatched_samples:
            # Safely get the list index for this sample
            s_id = sample.get('sample_idx')
            if s_id not in idx_map:
                continue
                
            idx = s_id - 1 # idx_map[s_id], idx in json file is 1+ shifted
            
            # Identify the target emotion of the starting sample
            current_target = full_txt_output[idx].get('target_emotion')
            
            # Start finding the block range [start, end]
            start = idx
            end = idx
            
            # Search backwards
            while start > 0:
                prev_sample = full_txt_output[start - 1]
                if prev_sample.get('target_emotion') == current_target:
                    start -= 1
                else:
                    break
            
            # Search forwards
            while end < total_samples - 1:
                next_sample = full_txt_output[end + 1]
                if next_sample.get('target_emotion') == current_target:
                    end += 1
                else:
                    break
            
            # Extract the block
            # slice is [start : end + 1] because the end index is exclusive in slicing
            block = full_txt_output[start : end + 1]
            
            # Add to storage
            block_dict[error_key].append(block)
            
    return block_dict

In [17]:
testdict = dict.fromkeys(grouped_data['Llama3']['FSL-24'].keys(), list())

In [34]:
def categorize_and_analyze_blocks(block_dict: Dict, total_samples: int, mismatched_dict: Dict, filepath: str):
    """
    Categorize blocks based on correct predictions in the sequence.
    
    Categories:
    - based_on_first: First turn is correct
    - based_on_last: Last turn is correct
    - based_on_others: Any middle turn is correct (but not first/last)
    - others: No turns are correct (or purely wrong block context)
    
    Also prints statistics and writes categorized blocks to file.
    """
    
    # Initialize storage lists
    based_on_first = []
    based_on_last = []
    based_on_others = []
    others = []
    
    # Initialize trackers
    # arraylist to track processed samples to avoid duplicate block processing
    processed_indices = np.zeros(total_samples + 1, dtype=int)
    
    # 1. Collect total number of mismatched data (from mismatched_dict/tleft0)
    total_mismatched_count = 0
    for key, samples in mismatched_dict.items():
        total_mismatched_count += len(samples)
        
    print(f"Total Mismatched Cases: {total_mismatched_count}")
    
    # Iterate through the block dictionary
    # key is (target_emo, pred_emo)
    for key, blocks in block_dict.items():
        target_emo, pred_emo = key
        
        for block in blocks:
            if not block: continue
            
            # check if this block has been processed
            # We check the first sample's index. 
            # If any sample in the block was processed, the whole block likely was.
            first_idx = block[0].get('sample_idx')
            if first_idx is not None and processed_indices[first_idx] == 1:
                continue
            
            # Mark all samples in this block as processed
            for turn in block:
                idx = turn.get('sample_idx')
                if idx is not None and idx < len(processed_indices):
                    processed_indices[idx] = 1
            
            # Analyze the block
            is_first_correct = block[0].get('is_correct', False)
            is_last_correct = block[-1].get('is_correct', False)
            
            # Check for "medium correct" (any middle turn is correct)
            is_medium_correct = False
            if len(block) > 2:
                for turn in block[1:-1]:
                    if turn.get('is_correct', False):
                        is_medium_correct = True
                        break
            
            if is_first_correct:
                based_on_first.append((key, block))
            elif is_last_correct:
                based_on_last.append((key, block))
            elif is_medium_correct:
                based_on_others.append((key, block))
            else:
                others.append((key, block))

    # 2. Print Output Statistics
    # Calculate duplicates removal impact?
    # Total unique blocks processed
    total_unique_blocks = len(based_on_first) + len(based_on_last) + len(based_on_others) + len(others)
    
    print("\nanalysis Statistics:")
    print(f"Total Unique Blocks Processed: {total_unique_blocks}")
    
    stats = [
        ("Based on First (First correct)", len(based_on_first)),
        ("Based on Last (Last correct)", len(based_on_last)),
        ("Based on Others (Middle correct)", len(based_on_others)),
        ("Others (All wrong)", len(others))
    ]
    
    for label, count in stats:
        # Comparison with overall mismatched cases might be tricky because one block can contain multiple mismatches,
        # or one mismatch is one block. 
        # Percentage relative to mismatched cases:
        pct = (count / total_unique_blocks) * 100 if total_unique_blocks > 0 else 0
        print(f"{label}: {count} ({pct:.2f}% of total mismatched cases)")

    # 3. Write to file
    output_filename = f'block_analysis_categorized.txt'
    all_wrong_filename = f'block_analysis_all_wrong.txt'
    # Use current directory or specific dir
    all_wrong_path = os.path.join(filepath, all_wrong_filename)
    filepath = os.path.join(filepath, output_filename)

    print(f"\nWriting categorized blocks to: {filepath}")
    
    def write_category_section(f, category_name, data_list):
        f.write(f"\n{'='*30} {category_name} (Count: {len(data_list)}) {'='*30}\n\n")
        
        for idx, ((target_emo, pred_emo), block) in enumerate(data_list):
            f.write(f"=== No.{idx+1} Block Misclassification: {target_emo} -> {pred_emo} ===\n")
            
            # Find the specific mismatch turn(s) vs correct turns to display
            # The user requested specific format for the text output
            
            for turn in block:
                sample_idx = turn.get('sample_idx', 'N/A')
                text = turn.get('text', '')
                t_emo = turn.get('target_emotion')
                p_emo = turn.get('pred_emotion')
                is_correct = turn.get('is_correct')
                
                # Mark if this is the error turn matching the key (target, pred)
                # Note: a block might have multiple errors.
                # The user's format:
                # [Sample 82] -- [New idx 0]
                # [target emotion: afraid -- pred emotion: terrified] ✗
                # [Input Text]...
                
                result_mark = "✓" if is_correct else "✗"
                
                f.write(f"[Sample {sample_idx}]\n")
                f.write(f"[target emotion: {t_emo} -- pred emotion: {p_emo}] {result_mark}\n")
                f.write(f"[Input Text]\n{text}\n")
                f.write("-" * 70 + "\n")
            f.write("\n")

    with open(filepath, mode='w', encoding='utf-8') as f:
        write_category_section(f, "Based on First (First Turn Correct)", based_on_first)
        write_category_section(f, "Based on Last (Last Turn Correct)", based_on_last)
        write_category_section(f, "Based on Others (Middle Turn Correct)", based_on_others)
        
    with open(all_wrong_path, mode='w', encoding='utf-8') as f:
        write_category_section(f, "Others (Entirely Wrong)", others)

    return based_on_first, based_on_last, based_on_others, others



In [35]:
# Run the analysis on the test data (assuming 'tright0' and 'block_dict_test' and 'tleft0' exist from previous cells)
for block_data in extracted_data_storage:
    output_dict, current_json, filepath = block_data
    dir_pathed = '/'.join(filepath.split('/')[:-1])
    blocked_dict = extract_dialogue_blocks(output_dict, current_json['text_output'])
    print(dir_pathed)
    
    categorize_and_analyze_blocks(
        blocked_dict, 
        current_json['total_samples'], 
        output_dict,
        dir_pathed
    )
else:
    print("Prerequisites missing: run previous cells to generate 'block_dict_test', 'tright0', 'tleft0'")

../outputs/Llama_02-11/method_FSL_bs_4_inputdata_ws_prompt_Llama3.3_FSL24
Total Mismatched Cases: 2205

analysis Statistics:
Total Unique Blocks Processed: 1205
Based on First (First correct): 178 (14.77% of total mismatched cases)
Based on Last (Last correct): 157 (13.03% of total mismatched cases)
Based on Others (Middle correct): 3 (0.25% of total mismatched cases)
Others (All wrong): 867 (71.95% of total mismatched cases)

Writing categorized blocks to: ../outputs/Llama_02-11/method_FSL_bs_4_inputdata_ws_prompt_Llama3.3_FSL24/block_analysis_categorized.txt
../outputs/Qwen3_02-11/method_SSP_bs_6_inputdata_ws_prompt_Qwen1d7_SSP_01
Total Mismatched Cases: 969

analysis Statistics:
Total Unique Blocks Processed: 540
Based on First (First correct): 60 (11.11% of total mismatched cases)
Based on Last (Last correct): 78 (14.44% of total mismatched cases)
Based on Others (Middle correct): 1 (0.19% of total mismatched cases)
Others (All wrong): 401 (74.26% of total mismatched cases)

Writin

## Semantic Similarity Analysis for Misclassifications

This section analyzes the "severity" of misclassifications using an embedding model.
- **Reasonable Error**: Target and Predicted emotions are semantically similar (e.g., afraid vs. terrified).
- **Significant Fault**: Target and Predicted emotions are semantically distant (e.g., afraid vs. angry).

We use `sentence-transformers` to compute embeddings and cosine similarity.


In [49]:
from sentence_transformers import SentenceTransformer, util
import pandas as pd

# 1. Initialize the Embedding Model on CUDA:3
device_id = 3
device = f'cuda:{device_id}' if torch.cuda.is_available() else 'cpu'

print(f"Loading SentenceTransformer model on {device}...")
# 'all-MiniLM-L6-v2' is a fast and effective model for semantic similarity
# You can also use 'all-mpnet-base-v2' for higher accuracy
embed_model = SentenceTransformer('BAAI/bge-base-en-v1.5', device=device)

# 2. Pre-compute embeddings for all emotions (Optimization)
# EMOTION_LIST is defined in previous cells
emotion_embeddings = embed_model.encode(EMOTION_LIST, convert_to_tensor=True, device=device)
emotion_map = {emo: idx for idx, emo in enumerate(EMOTION_LIST)}

print("Model loaded and emotion embeddings computed.")


Loading SentenceTransformer model on cuda:3...
Model loaded and emotion embeddings computed.


In [52]:
def evaluate_misclassification_severity(grouped_data: Dict, threshold: float = 0.5):
    """
    Evaluate misclassifications based on semantic similarity using the loaded embedding model.
    
    Args:
        grouped_data: collected misclassification data
        threshold: Cosine similarity threshold. 
                   Scores >= threshold are "Reasonable Errors"
                   Scores < threshold are "Significant Faults"
    """
    severity_report = {}
    
    print(f"Evaluating Semantic Severity (Threshold: {threshold})...")
    
    for model_name, strategies in grouped_data.items():
        severity_report[model_name] = {}
        
        # Skip metadata keys like 'filepath' directly if present, though loop usually handles sub-dicts
        # The structure is grouped_data[model][strategy] = {(target, pred): count}
        # But grouped_data[model] also has 'filepath'
        
        for strategy, errors in strategies.items():
            if not isinstance(errors, dict) or strategy == 'filepath': 
                continue
                
            total_errors = sum(errors.values())
            reasonable_count = 0
            significant_count = 0
            
            # Store details for visualization/inspection
            reasonable_examples = []
            significant_examples = []
            
            for (target, pred), count in errors.items():
                # Get indices
                if target not in emotion_map or pred not in emotion_map:
                    continue
                    
                idx_t = emotion_map[target]
                idx_p = emotion_map[pred]
                
                # Compute Similarity
                # embeddings are normalized, so dot product is cosine similarity
                emb_t = emotion_embeddings[idx_t]
                emb_p = emotion_embeddings[idx_p]
                similarity = util.cos_sim(emb_t, emb_p).item()
                
                if similarity >= threshold:
                    reasonable_count += count
                    reasonable_examples.append((target, pred, count, similarity))
                else:
                    significant_count += count
                    significant_examples.append((target, pred, count, similarity))
            
            # Sort examples by count desc
            reasonable_examples.sort(key=lambda x: x[2], reverse=True)
            significant_examples.sort(key=lambda x: x[2], reverse=True)
            
            severity_report[model_name][strategy] = {
                'total': total_errors,
                'reasonable': reasonable_count,
                'significant': significant_count,
                'reasonable_ratio': reasonable_count / total_errors if total_errors > 0 else 0,
                'significant_ratio': significant_count / total_errors if total_errors > 0 else 0,
                'top_reasonable': reasonable_examples[:20],
                'top_significant': significant_examples[:20]
            }
            
    return severity_report

def print_severity_analysis(report: Dict):
    print("\n" + "="*80)
    print("SENTIMENTAL MISCLASSIFICATION SEVERITY ANALYSIS")
    print("="*80)
    
    for model in report:
        print(f"\nModel: {model}")
        for strategy, stats in report[model].items():
            print(f"  Strategy: {strategy}")
            print(f"    Total Misclassifications: {stats['total']}")
            print(f"    - Reasonable Errors (High Similarity): {stats['reasonable']} ({stats['reasonable_ratio']:.1%})")
            print(f"    - Significant Faults (Low Similarity): {stats['significant']} ({stats['significant_ratio']:.1%})")
            
            print("    Top Reasonable Errors (Target -> Pred):")
            for t, p, c, s in stats['top_reasonable']:
                print(f"      {t} -> {p}: {c} times (Sim: {s:.4f})")
                
            print("    Top Significant Faults (Target -> Pred):")
            for t, p, c, s in stats['top_significant']:
                print(f"      {t} -> {p}: {c} times (Sim: {s:.4f})")
        print("-" * 60)

# Run the analysis
# Use grouped_data from previous cells
severity_report = evaluate_misclassification_severity(grouped_data, threshold=0.6)
print_severity_analysis(severity_report)


Evaluating Semantic Severity (Threshold: 0.6)...

SENTIMENTAL MISCLASSIFICATION SEVERITY ANALYSIS

Model: Llama3
  Strategy: FSL-24
    Total Misclassifications: 2205
    - Reasonable Errors (High Similarity): 1854 (84.1%)
    - Significant Faults (Low Similarity): 351 (15.9%)
    Top Reasonable Errors (Target -> Pred):
      terrified -> afraid: 94 times (Sim: 0.8670)
      anticipating -> excited: 77 times (Sim: 0.7435)
      angry -> furious: 74 times (Sim: 0.8772)
      guilty -> ashamed: 55 times (Sim: 0.6657)
      anxious -> afraid: 47 times (Sim: 0.7485)
      sad -> devastated: 43 times (Sim: 0.7692)
      afraid -> terrified: 41 times (Sim: 0.8670)
      nostalgic -> sentimental: 39 times (Sim: 0.7211)
      proud -> impressed: 38 times (Sim: 0.7235)
      lonely -> sad: 37 times (Sim: 0.7216)
      joyful -> excited: 34 times (Sim: 0.7367)
      proud -> joyful: 34 times (Sim: 0.6887)
      angry -> annoyed: 33 times (Sim: 0.8288)
      impressed -> surprised: 32 times (Sim:

In [53]:
def extract_and_save_significant_faults(grouped_data: Dict, threshold: float = 0.39):
    """
    Extracts 'Significant Faults' (low semantic similarity) and saves them to text files.
    Iterates through the original prediction files to find the actual text samples.
    """
    
    print(f"Extracting Significant Faults (Similarity < {threshold})...")
    
    for model_name, strategies in grouped_data.items():
        # Get filepath from the metadata in grouped_data if possible, 
        # but the structure is grouped_data[model][strategy]... 
        # We need the 'filepath' key which is at grouped_data[model]['filepath']
        
        filepath = strategies.get('filepath')
        if not filepath:
            print(f"Skipping {model_name}: No filepath found in grouped_data.")
            continue
            
        # Load the full predictions once per model
        try:
            full_preds = json_read(filepath)
            text_output = full_preds.get('text_output', [])
        except Exception as e:
            print(f"Error reading {filepath}: {e}")
            continue
            
        for strategy, errors in strategies.items():
            if not isinstance(errors, dict) or strategy == 'filepath':
                continue
                
            # IDENTIFY SIGNIFICANT PAIRS
            significant_pairs = set()
            for (target, pred), count in errors.items():
                if target not in emotion_map or pred not in emotion_map:
                    continue
                idx_t = emotion_map[target]
                idx_p = emotion_map[pred]
                similarity = util.cos_sim(emotion_embeddings[idx_t], emotion_embeddings[idx_p]).item()
                
                if similarity < threshold:
                    significant_pairs.add((target, pred))
            
            if not significant_pairs:
                print(f"No significant faults found for {model_name} - {strategy}")
                continue
                
            # COLLECT SAMPLES
            # We iterate through the FULL text output to find samples matching our significant pairs
            # This preserves the original order and context
            significant_samples = []
            
            for sample in text_output:
                if sample.get('is_correct', True): continue
                
                t = sample.get('target_emotion')
                p = sample.get('pred_emotion')
                
                if (t, p) in significant_pairs:
                    # Calculate similarity again for display (or cache it)
                    # For speed, re-calc is negligible here
                    sim = util.cos_sim(
                        emotion_embeddings[emotion_map[t]], 
                        emotion_embeddings[emotion_map[p]]
                    ).item()
                    
                    sample['_sim'] = sim # Inject sim for printing
                    significant_samples.append(sample)
            
            # SORT SAMPLES
            # Option 1: Sort by Similarity (Lowest first - most severe)
            # Option 2: Sort by Frequency of that pair (as in previous code)
            # Let's sort by Similarity ascending (Most severe errors first)
            significant_samples.sort(key=lambda x: x['_sim'])
            
            # WRITE TO FILE
            file_dir = os.path.dirname(filepath)
            file_name = f'significant_faults_{model_name}_{strategy}.txt'
            out_path = os.path.join(file_dir, file_name)
            
            print(f"Writing {len(significant_samples)} significant faults to: {out_path}")
            
            with open(out_path, mode='w', encoding='utf-8') as f:
                f.write(f"=== SIGNIFICANT FAULTS REPORT ===\n")
                f.write(f"Model: {model_name} | Strategy: {strategy}\n")
                f.write(f"Threshold: {threshold} (Similarity < {threshold})\n")
                f.write(f"Total Significant Samples: {len(significant_samples)}\n")
                f.write("="*70 + "\n\n")
                
                for i, sample in enumerate(significant_samples):
                    t = sample.get('target_emotion')
                    p = sample.get('pred_emotion')
                    sim = sample.get('_sim')
                    text = sample.get('text', '')
                    idx = sample.get('sample_idx', 'N/A')
                    
                    f.write(f"--- Sample {idx} (Severity Rank: {i+1}) ---\n")
                    f.write(f"Target: {t} | Pred: {p}\n")
                    f.write(f"Similarity: {sim:.4f} (Low)\n")
                    f.write(f"[Input Text]:\n{text}\n")
                    f.write("\n")

# Execute extraction
if 'grouped_data' in locals() and 'emotion_embeddings' in locals():
    extract_and_save_significant_faults(grouped_data, threshold=0.39)
else:
    print("Please ensure 'grouped_data' and embedding models are loaded.")


Extracting Significant Faults (Similarity < 0.39)...
No significant faults found for Llama3 - FSL-24
No significant faults found for Qwen4B - FSL-32
No significant faults found for Qwen4B - SSL-0.4
No significant faults found for Qwen1.7B - SSL-0.5
